# Аналитическая модель: структурный разбор данных

## Что я вижу в этих данных

После детального EDA обнаружены три нетривиальных паттерна:

### 1. Paid rate — скрытый системный показатель
Отношение **Платные/Выступления** является КОНСТАНТОЙ внутри каждого года:
- 2024: **63.0%** (все 4 квартала — точное совпадение)
- 2025: **54.0%** (все 4 квартала — точное совпадение)
- Q1 2026: **41.1%** — новый уровень

Это не случайность. Это **системное изменение**: ценовая политика или доля бесплатных мероприятий меняется раз в год. Тренд: −9 п.п./год.

### 2. Выступления и Платные — одна метрика
**Платные = Выступления × paid_rate**. Прогнозировать их независимо бессмысленно.

### 3. Оплаты — нелинейный сбор с двумя аномалиями
Collection rate (Оплаты/Платные) нестабилен: 20–92% по кварталам.
- Q3 2025 = **91.9%** — сбор накопленного долга
- Q1 2026 = **74.2%** — авансовые платежи или погашение задолженности
- Q1 2026 в **3.49×** выше Q1 2025 — аномалия, которую нельзя экстраполировать напрямую

### Архитектура модели
```
Выступления:  YoY-тренд(взвеш.) × сезонный профиль 2025
Платные:      Выступления × paid_rate_2026  (paid_rate по тренду+Q1 сигнал)
Оплаты:       ограниченный годовой прогноз × сезонные доли, с Q1-авансовой поправкой
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 130

quarters = [
    'Q1 2024', 'Q2 2024', 'Q3 2024', 'Q4 2024',
    'Q1 2025', 'Q2 2025', 'Q3 2025', 'Q4 2025',
    'Q1 2026',
]
raw = {
    'Выступления':         [1595, 8913, 16406, 27472, 9780, 18358, 10499, 28077, 12350],
    'Платные выступления': [1005, 5615, 10336, 17307, 5281,  9913,  5669, 15162,  5070],
    'Оплаты':              [ 365, 2384,  2970,  9668, 1077,  5548,  5212, 10210,  3761],
}
B = np.array(raw['Выступления'], float)
P = np.array(raw['Платные выступления'], float)
O = np.array(raw['Оплаты'], float)

n_hist            = 9
forecast_quarters = ['Q2 2026', 'Q3 2026', 'Q4 2026']
all_quarters      = quarters + forecast_quarters

print('Данные загружены.')

In [ ]:
# ── EDA: ключевые паттерны ────────────────────────────────────────────────────
paid_rate = P / B
coll_rate = O / P

fig = plt.figure(figsize=(16, 11))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.38)

x   = np.arange(n_hist)
xl  = [q.replace(' ', '\n') for q in quarters]

# 1. Paid rate
ax1 = fig.add_subplot(gs[0, :2])
bar_colors = ['#2563EB']*4 + ['#16A34A']*4 + ['#DC2626']
ax1.bar(x, paid_rate * 100, color=bar_colors, alpha=0.85)
for xi, r in zip(x, paid_rate):
    ax1.text(xi, r*100 + 0.5, f'{r*100:.1f}%', ha='center', fontsize=9, fontweight='bold')
ax1.axhline(63.0, color='#2563EB', ls='--', lw=1.5, label='2024 уровень (63%)')
ax1.axhline(54.0, color='#16A34A', ls='--', lw=1.5, label='2025 уровень (54%)')
ax1.axhline(41.1, color='#DC2626', ls='--', lw=1.5, alpha=0.7, label='Q1 2026 (41.1%)')
ax1.set_xticks(x); ax1.set_xticklabels(xl, fontsize=8)
ax1.set_ylabel('Paid rate (%)')
ax1.set_title('PAID RATE = Платные / Выступления\n→ Константа внутри года, системное снижение −9 п.п./год', fontsize=11, fontweight='bold')
ax1.legend(fontsize=8.5); ax1.set_ylim(0, 80); ax1.grid(axis='y', alpha=0.3)

# 2. Collection rate
ax2 = fig.add_subplot(gs[0, 2])
bar_colors2 = ['#6B7280']*4 + ['#F59E0B']*4 + ['#EF4444']
ax2.bar(x, coll_rate * 100, color=bar_colors2, alpha=0.85)
for xi, r in zip(x, coll_rate):
    ax2.text(xi, r*100 + 2, f'{r*100:.0f}%', ha='center', fontsize=8, fontweight='bold')
ax2.set_xticks(x); ax2.set_xticklabels(xl, fontsize=7)
ax2.set_title('COLLECTION RATE\n= Оплаты / Платные', fontsize=11, fontweight='bold')
ax2.set_ylabel('Collection rate (%)')
ax2.grid(axis='y', alpha=0.3)
ax2.annotate('Сбор долгов?', xy=(6, 91.9), xytext=(5.2, 108),
             arrowprops=dict(arrowstyle='->', color='red'),
             ha='center', fontsize=8, color='red')
ax2.annotate('Авансы?', xy=(8, 74.2), xytext=(7.2, 90),
             arrowprops=dict(arrowstyle='->', color='darkred'),
             ha='center', fontsize=8, color='darkred')

# 3. YoY
ax3 = fig.add_subplot(gs[1, :])
yoy_B = B[4:] / B[:5]
yoy_P = P[4:] / P[:5]
yoy_O = O[4:] / O[:5]
yoy_labels = ['Q1\n(2025/2024)', 'Q2\n(2025/2024)', 'Q3\n(2025/2024)', 'Q4\n(2025/2024)', 'Q1\n(2026/2025)']
xq = np.arange(5); w = 0.25
ax3.bar(xq - w, yoy_B, w, color='#2563EB', label='Выступления', alpha=0.85)
ax3.bar(xq,     yoy_P, w, color='#16A34A', label='Платные',     alpha=0.85)
ax3.bar(xq + w, yoy_O, w, color='#DC2626', label='Оплаты',      alpha=0.85)
ax3.axhline(1.0, color='gray', ls='--', lw=1.5)
ax3.set_xticks(xq); ax3.set_xticklabels(yoy_labels, fontsize=10)
ax3.set_ylabel('YoY коэффициент роста')
ax3.set_title('YoY рост по кварталам\n→ Q1 2026 аномально высок по Оплатам (3.49×) — нельзя экстраполировать прямо', fontsize=11, fontweight='bold')
ax3.legend(fontsize=10); ax3.grid(axis='y', alpha=0.3)

for xi, v in zip(xq + w, yoy_O):
    ax3.text(xi, v + 0.07, f'{v:.2f}×', ha='center', fontsize=9, fontweight='bold', color='#DC2626')

plt.suptitle('EDA: структурные паттерны в данных', fontsize=14, fontweight='bold')
plt.savefig('forecast_2026_analytical_eda.png', bbox_inches='tight', dpi=150)
plt.show()
print('EDA-график сохранён.')

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# АНАЛИТИЧЕСКАЯ МОДЕЛЬ
# ════════════════════════════════════════════════════════════════════════════

y24_B = B[:4]; y25_B = B[4:8]; q1_26_B = B[8]
y24_P = P[:4]; y25_P = P[4:8]; q1_26_P = P[8]
y24_O = O[:4]; y25_O = O[4:8]; q1_26_O = O[8]

annual_24_B = y24_B.sum(); annual_25_B = y25_B.sum()
annual_24_P = y24_P.sum(); annual_25_P = y25_P.sum()
annual_24_O = y24_O.sum(); annual_25_O = y25_O.sum()

# ── Блок 1: ВЫСТУПЛЕНИЯ ──────────────────────────────────────────────────────
g_hist_B = annual_25_B / annual_24_B     # 1.227
g_q1_B   = q1_26_B / y25_B[0]           # 1.263
# 75% вес истории, 25% Q1-сигнал (одна точка менее надёжна, чем 4)
g_B = 0.75 * g_hist_B + 0.25 * g_q1_B

annual_26_B           = annual_25_B * g_B
annual_26_remaining_B = annual_26_B - q1_26_B
rem_shares_B          = y25_B[1:] / y25_B[1:].sum()
fc_B = annual_26_remaining_B * rem_shares_B

print('=== ВЫСТУПЛЕНИЯ ===')
print(f'  g_hist={g_hist_B:.3f}  g_Q1={g_q1_B:.3f}  g_взвеш={g_B:.3f}')
print(f'  Годовой 2026 = {annual_26_B:.0f}  |  остаток Q2-Q4 = {annual_26_remaining_B:.0f}')
print(f'  Прогноз: Q2={fc_B[0]:.0f}  Q3={fc_B[1]:.0f}  Q4={fc_B[2]:.0f}')

# ── Блок 2: ПЛАТНЫЕ (через paid_rate) ────────────────────────────────────────
paid_rate_24 = (P[:4] / B[:4]).mean()   # 0.630
paid_rate_25 = (P[4:8] / B[4:8]).mean() # 0.540
paid_rate_q1 = q1_26_P / q1_26_B        # 0.411
trend_pr     = paid_rate_25 - paid_rate_24  # -0.090
rate_extrap  = paid_rate_25 + trend_pr      # 0.450 (линейный тренд)

# 60% линейный тренд + 40% Q1-сигнал
# Q1 уже произошёл — важный сигнал, но paid_rate может быть специфичен для Q1
paid_rate_26 = 0.60 * rate_extrap + 0.40 * paid_rate_q1

fc_P = fc_B * paid_rate_26

print(f'\n=== ПЛАТНЫЕ ВЫСТУПЛЕНИЯ ===')
print(f'  paid_rate: 2024={paid_rate_24:.3f}  2025={paid_rate_25:.3f}  Q1_2026={paid_rate_q1:.3f}')
print(f'  rate_extrap={rate_extrap:.3f}  paid_rate_2026={paid_rate_26:.3f}')
print(f'  Прогноз: Q2={fc_P[0]:.0f}  Q3={fc_P[1]:.0f}  Q4={fc_P[2]:.0f}')

# ── Блок 3: ОПЛАТЫ ───────────────────────────────────────────────────────────
# Два якоря для годового объёма:
g_O_hist = annual_25_O / annual_24_O             # 1.43 — исторический тренд
annual_26_O_trend = annual_25_O * g_O_hist       # 31 527

# Q1-based якорь с жёстким cap:
# Максимум доверяем росту в 2× к историческому тренду (без cap получается 77k)
q1_share_O     = y25_O[0] / annual_25_O
annual_26_O_q1 = q1_26_O / q1_share_O           # 76 990 — слишком много
annual_26_O_q1_capped = min(annual_26_O_q1, annual_26_O_trend * 1.80)

# Геом. среднее ограниченных якорей
annual_26_O = np.sqrt(annual_26_O_trend * annual_26_O_q1_capped)

# Авансовая поправка: Q1 аномально превысил ожидания
q1_expected_O   = y25_O[0] * g_O_hist
q1_overcollect  = max(0, q1_26_O - q1_expected_O)
# 25% "аванса" вычтем из Q2 (постепенно рассасывается)
advance_adj = q1_overcollect * 0.25

annual_26_remaining_O = annual_26_O - q1_26_O
rem_shares_O = y25_O[1:] / y25_O[1:].sum()
fc_O         = annual_26_remaining_O * rem_shares_O

fc_O[0] = max(fc_O[0] - advance_adj, fc_O[0] * 0.80)
fc_O    = np.maximum(fc_O, 0)

print(f'\n=== ОПЛАТЫ ===')
print(f'  Якорь A (тренд): {annual_26_O_trend:.0f}')
print(f'  Якорь B (Q1, raw): {annual_26_O_q1:.0f}  → cap: {annual_26_O_q1_capped:.0f}')
print(f'  Годовой 2026 (геом. среднее): {annual_26_O:.0f}')
print(f'  Q1 ожидался: {q1_expected_O:.0f}  фактический: {q1_26_O:.0f}  "аванс": {q1_overcollect:.0f}')
print(f'  Поправка Q2: −{advance_adj:.0f}')
print(f'  Прогноз: Q2={fc_O[0]:.0f}  Q3={fc_O[1]:.0f}  Q4={fc_O[2]:.0f}')

In [ ]:
# ── Диапазоны неопределённости ────────────────────────────────────────────────

# Выступления: g от g_hist до g_q1
fc_B_lo = (annual_25_B * g_hist_B - q1_26_B) * rem_shares_B
fc_B_hi = (annual_25_B * g_q1_B   - q1_26_B) * rem_shares_B

# Платные: paid_rate от Q1-сигнала до rate_extrap
rate_lo = min(paid_rate_q1, rate_extrap)
rate_hi = max(paid_rate_q1, rate_extrap)
fc_P_lo = fc_B * rate_lo
fc_P_hi = fc_B * rate_hi

# Оплаты: ±20% от прогноза
fc_O_lo = fc_O * 0.80
fc_O_hi = fc_O * 1.20

print('=' * 80)
print('АНАЛИТИЧЕСКАЯ МОДЕЛЬ: прогноз Q2–Q4 2026')
print('=' * 80)
print(f'{"Метрика":<25} {"Квартал":<10} {"Прогноз":>9} {"Мин":>9} {"Макс":>9}')
print('-' * 80)

for fc, lo, hi, name in [
    (fc_B, fc_B_lo, fc_B_hi, 'Выступления'),
    (fc_P, fc_P_lo, fc_P_hi, 'Платные выступления'),
    (fc_O, fc_O_lo, fc_O_hi, 'Оплаты'),
]:
    lo_c = np.maximum(lo, 0)
    hi_c = np.maximum(hi, 0)
    for i, q in enumerate(['Q2 2026', 'Q3 2026', 'Q4 2026']):
        print(f"{name:<25} {q:<10} {int(round(fc[i])):>9,} {int(round(lo_c[i])):>9,} {int(round(hi_c[i])):>9,}".replace(',', ' '))

In [ ]:
# ── Главный прогнозный график ────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 16), sharex=True)
fig.suptitle(
    'Аналитическая модель Q2–Q4 2026\n'
    '(paid_rate тренд  ·  взвешенный YoY  ·  Q1-авансовая поправка)',
    fontsize=14, fontweight='bold', y=0.999)

x_all  = np.arange(len(all_quarters))
x_hist = x_all[:n_hist]
x_fc   = x_all[n_hist:]

plot_cfg = [
    ('Выступления',         '#2563EB', B,
     fc_B, np.maximum(fc_B_lo, 0), fc_B_hi,
     f'g={g_B:.2f} (hist={g_hist_B:.2f}, Q1={g_q1_B:.2f})'),
    ('Платные выступления', '#16A34A', P,
     fc_P, np.maximum(fc_P_lo, 0), fc_P_hi,
     f'paid_rate={paid_rate_26:.2f} (экстрап={rate_extrap:.2f}, Q1={paid_rate_q1:.2f})'),
    ('Оплаты',              '#DC2626', O,
     fc_O, np.maximum(fc_O_lo, 0), fc_O_hi,
     f'годовой={annual_26_O:.0f} (Q1-аванс −{advance_adj:.0f})'),
]

for ax, (metric, mc, y_hist, fc, lo, hi, subtitle) in zip(axes, plot_cfg):
    ax.scatter(x_hist, y_hist, color=mc, s=70, zorder=6)
    ax.plot(x_hist, y_hist, '-', color=mc, lw=1.3, alpha=0.5)

    ax.fill_between(x_fc, lo, hi, color=mc, alpha=0.15, label='Диапазон')
    ax.plot(x_fc, lo, '--', color=mc, lw=1.2, alpha=0.6)
    ax.plot(x_fc, hi, '--', color=mc, lw=1.2, alpha=0.6)
    ax.plot(x_fc, fc, 'o-', color=mc, lw=2.8, ms=9, zorder=5, label='Прогноз')

    for xi, yi in zip(x_fc, fc):
        ax.annotate(f'{int(round(yi)):,}'.replace(',', '\u202f'),
                    xy=(xi, yi), xytext=(0, 11), textcoords='offset points',
                    ha='center', fontsize=10, fontweight='bold', color=mc)

    ax.axvline(x=n_hist - 0.5, color='gray', ls=':', lw=1.5)
    ylo_ax, yhi_ax = ax.get_ylim()
    ax.text(n_hist - 0.38, yhi_ax * 0.97, 'прогноз ->', fontsize=8, color='gray', va='top')

    ax.set_title(f'{metric}   [{subtitle}]', fontsize=10.5, fontweight='bold', pad=6)
    ax.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda x, _: f'{int(x):,}'.replace(',', '\u202f'))
    )
    ax.grid(axis='y', alpha=0.3)
    ax.legend(loc='upper left', fontsize=9, framealpha=0.9)

axes[-1].set_xticks(x_all)
axes[-1].set_xticklabels(all_quarters, rotation=35, ha='right', fontsize=10)

plt.tight_layout()
plt.savefig('forecast_2026_analytical.png', bbox_inches='tight', dpi=150)
plt.show()
print('График сохранён: forecast_2026_analytical.png')

In [ ]:
# ── Визуализация paid_rate тренда + Оплаты якоря ─────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Системный анализ ключевых коэффициентов', fontsize=13, fontweight='bold')

# Paid rate
ax1.plot([2024, 2025], [paid_rate_24*100, paid_rate_25*100],
         'o-', color='#374151', lw=2, ms=10, zorder=3, label='Факт')
ax1.scatter([2026], [paid_rate_26*100], color='#F59E0B', s=150,
            marker='D', zorder=6, label=f'Прогноз 2026 ({paid_rate_26*100:.1f}%)')
ax1.scatter([2026], [rate_extrap*100], color='#6B7280', s=80,
            marker='^', zorder=5, alpha=0.7, label=f'Линейный тренд ({rate_extrap*100:.1f}%)')
ax1.scatter([2026], [paid_rate_q1*100], color='#DC2626', s=80,
            marker='v', zorder=5, alpha=0.7, label=f'Q1 2026 сигнал ({paid_rate_q1*100:.1f}%)')

for yr, rt, c in [(2024, paid_rate_24*100, '#2563EB'),
                   (2025, paid_rate_25*100, '#16A34A'),
                   (2026, paid_rate_26*100, '#F59E0B')]:
    ax1.annotate(f'{rt:.1f}%', xy=(yr, rt), xytext=(0, 10),
                 textcoords='offset points', ha='center',
                 fontsize=12, fontweight='bold', color=c)

ax1.set_xlim(2023.5, 2026.8); ax1.set_xticks([2024, 2025, 2026])
ax1.set_ylabel('Paid rate (%)'); ax1.set_ylim(30, 72)
ax1.set_title('Paid Rate: системное снижение\n−9 п.п./год', fontweight='bold')
ax1.legend(fontsize=8.5); ax1.grid(alpha=0.3)

# Оплаты якоря
labels_O = ['Якорь A\n(историч. тренд)', 'Якорь B\n(Q1-cap)', 'Прогноз\n(геом. среднее)']
vals_O   = [annual_26_O_trend, annual_26_O_q1_capped, annual_26_O]
colors_O = ['#6B7280', '#EF4444', '#F59E0B']
bars = ax2.bar(labels_O, vals_O, color=colors_O, alpha=0.85, width=0.5)
for bar, val in zip(bars, vals_O):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 500,
             f'{int(round(val)):,}'.replace(',', '\u202f'),
             ha='center', fontsize=11, fontweight='bold')
ax2.axhline(annual_25_O, color='#16A34A', ls='--', lw=1.8,
            label=f'2025 факт ({annual_25_O:,.0f})'.replace(',', '\u202f'))
ax2.set_title('Годовой прогноз Оплат 2026\n(два якоря + геом. среднее)', fontweight='bold')
ax2.set_ylabel('Сумма Оплат')
ax2.legend(fontsize=9)
ax2.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}'.replace(',', '\u202f')))
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('forecast_2026_analytical_rates.png', bbox_inches='tight', dpi=150)
plt.show()
print('Коэффициентный график сохранён.')

In [ ]:
# ── Сравнение всех трёх моделей ──────────────────────────────────────────────
models = {
    'Референсная':   {
        'Выступления':         [22851, 13068, 34948],
        'Платные выступления': [ 9970,  5702, 15249],
        'Оплаты':              [ 8136,  7643, 14972],
    },
    'Ансамблевая':   {
        'Выступления':         [23008, 15511, 32383],
        'Платные выступления': [10630,  7381, 16279],
        'Оплаты':              [13415, 11052, 12291],
    },
    'Аналитическая': {
        'Выступления':         [int(round(v)) for v in fc_B],
        'Платные выступления': [int(round(v)) for v in fc_P],
        'Оплаты':              [int(round(v)) for v in fc_O],
    },
}

print('=' * 95)
print('СРАВНЕНИЕ ТРЁХ МОДЕЛЕЙ')
print('=' * 95)
print(f'{"Метрика":<25} {"Квартал":<10} {"Референсная":>14} {"Ансамблевая":>14} {"Аналитическая":>15}')
print('-' * 95)
for m in ['Выступления', 'Платные выступления', 'Оплаты']:
    for i, q in enumerate(['Q2 2026', 'Q3 2026', 'Q4 2026']):
        v = [models[mod][m][i] for mod in models]
        row = '   '.join(f'{x:>12,}'.replace(',', ' ') for x in v)
        print(f'{m:<25} {q:<10} {row}')
    print()

# Сводная: среднее Q2-Q4
print('Среднее по Q2–Q4:')
for m in ['Выступления', 'Платные выступления', 'Оплаты']:
    avgs = [int(np.mean(models[mod][m])) for mod in models]
    row  = '   '.join(f'{x:>12,}'.replace(',', ' ') for x in avgs)
    print(f'{m:<25} {"среднее":<10} {row}')

## Аналитические выводы

### Что аналитическая модель делает иначе

**Paid rate** — главная «находка». Обе предыдущие модели прогнозировали Платные и Выступления независимо и не заметили, что их отношение точно константа внутри года. Аналитическая модель явно прогнозирует снижение paid_rate до ~43% и применяет его к прогнозу Выступлений. Это даёт **более низкие Платные** (~9.8k в Q2 против 10–11k у других моделей).

**Оплаты Q1 2026** — ансамблевая модель принимает рост 3.49× как тренд, давая 13k+ в Q2–Q3. Аналитическая модель применяет cap на Q1-якоре и вычитает авансовую поправку — результат ~10–11k, что ближе к референсной (~8k). Истина скорее всего между ними.

### Ключевые риски прогноза
1. **Paid rate может не стабилизироваться** — если снижение продолжится до 32% к Q4, прогноз Платных завышен
2. **Природа Q1 2026 Оплат неясна** — если это реальный рост воронки, ансамблевая модель права; если разовый сбор долгов — аналитическая права
3. **Q4 всегда пиковый** (42% всего года) — самый надёжный квартал для прогноза

### Рекомендуемый диапазон планирования
| Метрика | Q2 2026 | Q3 2026 | Q4 2026 |
|---------|---------|---------|--------|
| Выступления | 22 600–23 000 | 13 000–15 500 | 33 000–35 000 |
| Платные выступления | 9 800–10 600 | 5 600–7 400 | 15 000–16 300 |
| Оплаты | 8 100–11 400 | 7 600–11 300 | 13 000–22 000 |